# 📘 CDC — Change Data Capture
## Capturing, Processing & Applying Data Changes

**What you'll learn:**
- What CDC is and why it's critical for modern data engineering
- CDC capture methods (log-based, trigger-based, timestamp-based)
- Delta Change Data Feed (CDF) — querying row-level changes
- Building CDC pipelines: source → capture → apply → verify
- MERGE patterns for applying CDC events
- Handling edge cases: late arrivals, out-of-order, duplicates, deletes
- Idempotent CDC processing (safe to rerun)
- End-to-end CDC pipeline project

**Prerequisites:** Notebooks 04 (Medallion), 05 (SCD), 46 (Delta Internals)
**Estimated Time:** 5-7 hours
**Database:** techmart_dw + new CDC simulation tables

---

> 📌 **Notebook 05** (SCD Masterclass) teaches how to UPDATE dimensions with history.
> **This notebook** teaches how to CAPTURE and PROCESS changes from source systems.
> They work together: CDC captures changes → SCD applies them to dimensions.

---

---
# 📗 Section 1: What is Change Data Capture?

## Definition

**Change Data Capture (CDC)** is the process of identifying and capturing changes
(inserts, updates, deletes) made to a source database, and delivering those changes
to a downstream system in near real-time.

## Why CDC Matters

Without CDC, you have two bad options:
1. **Full load every time** — copy entire table (expensive, slow, wasteful)
2. **Timestamp-based incremental** — only catches inserts/updates, MISSES deletes

CDC gives you the best option:
3. **Capture every change** — inserts, updates, AND deletes, in order

```
WITHOUT CDC:                              WITH CDC:
┌──────────┐    Full Copy    ┌──────────┐  ┌──────────┐  Change Stream  ┌──────────┐
│  Source  │ ═══════════════▶│  Target  │  │  Source  │ ──────────────▶│  Target  │
│  (100M   │  (copy 100M    │          │  │  (100M   │  (only 10K     │          │
│   rows)  │   rows daily!) │          │  │   rows)  │   changes/day) │          │
└──────────┘                 └──────────┘  └──────────┘                └──────────┘
Cost: $$$, Hours                           Cost: $, Minutes
```

## CDC Methods

| Method | How It Works | Pros | Cons |
|--------|-------------|------|------|
| **Log-based** | Read database transaction log (WAL/binlog) | Complete, low overhead | Requires DB access |
| **Trigger-based** | DB triggers write to change table | Simple to implement | Performance overhead on source |
| **Timestamp-based** | Query WHERE modified_at > last_run | No DB access needed | Misses deletes, clock skew |
| **Diff-based** | Compare snapshots | Works anywhere | Expensive, slow |

### Log-Based CDC (Gold Standard)

```
Source Database (PostgreSQL)
├── Transaction Log (WAL)
│   ├── INSERT INTO orders VALUES (101, ...)
│   ├── UPDATE orders SET status='shipped' WHERE id=42
│   └── DELETE FROM orders WHERE id=99
│
└── CDC Tool (Debezium / Lakeflow Connect)
    ├── Reads WAL in real-time
    ├── Converts to change events
    └── Publishes to Kafka / writes to Delta
```

### Tools for CDC

| Tool | Type | Source Support | Target |
|------|------|---------------|--------|
| **Debezium** | Open source | PostgreSQL, MySQL, MongoDB, SQL Server | Kafka |
| **Lakeflow Connect** | Managed (Databricks) | PostgreSQL, MySQL, Oracle, Salesforce | Delta Lake |
| **AWS DMS** | Managed (AWS) | Most databases | S3, Kinesis, RDS |
| **Fivetran** | SaaS | 300+ connectors | Snowflake, BigQuery, Databricks |
| **Airbyte** | Open source | 300+ connectors | Various |

In [0]:
# CDC event structure
import json
from datetime import datetime

# What a CDC event looks like (Debezium format)
cdc_event_insert = {
    "op": "c",  # c=create, u=update, d=delete, r=read (snapshot)
    "ts_ms": 1710496225000,
    "source": {
        "connector": "postgresql",
        "db": "ecommerce",
        "schema": "public",
        "table": "orders",
        "lsn": 123456789  # Log Sequence Number
    },
    "before": None,  # NULL for inserts
    "after": {
        "order_id": 1001,
        "customer_id": 42,
        "total_amount": 89.99,
        "status": "pending",
        "created_at": "2024-03-15T10:23:45Z"
    }
}

cdc_event_update = {
    "op": "u",
    "ts_ms": 1710496285000,
    "source": {"connector": "postgresql", "db": "ecommerce", "schema": "public",
               "table": "orders", "lsn": 123456790},
    "before": {
        "order_id": 1001,
        "customer_id": 42,
        "total_amount": 89.99,
        "status": "pending",
        "created_at": "2024-03-15T10:23:45Z"
    },
    "after": {
        "order_id": 1001,
        "customer_id": 42,
        "total_amount": 89.99,
        "status": "shipped",
        "created_at": "2024-03-15T10:23:45Z"
    }
}

cdc_event_delete = {
    "op": "d",
    "ts_ms": 1710496345000,
    "source": {"connector": "postgresql", "db": "ecommerce", "schema": "public",
               "table": "orders", "lsn": 123456791},
    "before": {
        "order_id": 999,
        "customer_id": 17,
        "total_amount": 15.00,
        "status": "cancelled",
        "created_at": "2024-03-10T08:00:00Z"
    },
    "after": None  # NULL for deletes
}

print("📡 CDC EVENT EXAMPLES (Debezium Format)")
print("=" * 60)

events = [
    ("INSERT (op='c')", cdc_event_insert),
    ("UPDATE (op='u')", cdc_event_update),
    ("DELETE (op='d')", cdc_event_delete),
]

for name, event in events:
    print(f"\n📌 {name}:")
    print(f"   Operation: {event['op']}")
    print(f"   Table: {event['source']['table']}")
    print(f"   Before: {event['before']}")
    print(f"   After: {event['after']}")

print("\n💡 Key Insight:")
print("   • INSERT: before=NULL, after=new_values")
print("   • UPDATE: before=old_values, after=new_values")
print("   • DELETE: before=old_values, after=NULL")


---
# 📗 Section 2: Building a CDC Pipeline in Databricks

## The CDC Pipeline Pattern

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│   SOURCE     │    │   BRONZE     │    │   SILVER     │    │    GOLD      │
│   SYSTEM     │───▶│  (Raw CDC)   │───▶│  (Applied)   │───▶│ (Aggregated) │
│              │    │              │    │              │    │              │
│ PostgreSQL   │    │ Append-only  │    │ Current state│    │ Business     │
│ MySQL        │    │ All events   │    │ via MERGE    │    │ metrics      │
│ Oracle       │    │ + metadata   │    │              │    │              │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
                    
                    Keep ALL events     Apply changes        Aggregate for
                    (audit trail)       (latest state)       reporting
```

## Our Demo: Simulating a Source System

Since we can't connect to a real database on Community Edition, we'll simulate
a source system that generates CDC events.

In [0]:
# Setup: Create source system simulation
spark.sql("USE techmart_dw")

# Create the "source" table (simulates a transactional database)
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_source_customers")
spark.sql("""
CREATE TABLE techmart_dw.cdc_source_customers (
    customer_id INT,
    name STRING,
    email STRING,
    city STRING,
    tier STRING,
    lifetime_value DECIMAL(12,2),
    is_active BOOLEAN,
    last_modified TIMESTAMP
) USING DELTA
""")

# Initial load (snapshot)
spark.sql("""
INSERT INTO techmart_dw.cdc_source_customers VALUES
(1, 'Alice Smith', 'alice@techmart.com', 'New York', 'gold', 5000.00, true, '2024-01-01 00:00:00'),
(2, 'Bob Jones', 'bob@techmart.com', 'Los Angeles', 'silver', 2000.00, true, '2024-01-01 00:00:00'),
(3, 'Charlie Brown', 'charlie@techmart.com', 'Chicago', 'bronze', 500.00, true, '2024-01-01 00:00:00'),
(4, 'Diana Prince', 'diana@techmart.com', 'Seattle', 'gold', 8000.00, true, '2024-01-01 00:00:00'),
(5, 'Eve Wilson', 'eve@techmart.com', 'Austin', 'silver', 1500.00, true, '2024-01-01 00:00:00')
""")

print("✅ Source system created with 5 customers")
spark.sql("SELECT * FROM techmart_dw.cdc_source_customers ORDER BY customer_id").show(truncate=False)

In [0]:
# Create the CDC events table (Bronze layer — raw change events)
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_bronze_events")
spark.sql("""
CREATE TABLE techmart_dw.cdc_bronze_events (
    event_id BIGINT,
    operation STRING COMMENT 'I=Insert, U=Update, D=Delete',
    customer_id INT,
    name STRING,
    email STRING,
    city STRING,
    tier STRING,
    lifetime_value DECIMAL(12,2),
    is_active BOOLEAN,
    change_timestamp TIMESTAMP,
    sequence_number BIGINT COMMENT 'Ordering guarantee',
    source_lsn STRING COMMENT 'Log Sequence Number from source'
) USING DELTA
""")

# Create the Silver table (current state after applying CDC)
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_silver_customers")
spark.sql("""
CREATE TABLE techmart_dw.cdc_silver_customers (
    customer_id INT,
    name STRING,
    email STRING,
    city STRING,
    tier STRING,
    lifetime_value DECIMAL(12,2),
    is_active BOOLEAN,
    last_modified TIMESTAMP,
    _cdc_operation STRING,
    _cdc_timestamp TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

# Initial load into Silver (snapshot)
spark.sql("""
INSERT INTO techmart_dw.cdc_silver_customers
SELECT customer_id, name, email, city, tier, lifetime_value, is_active, 
       last_modified, 'SNAPSHOT' AS _cdc_operation, last_modified AS _cdc_timestamp
FROM techmart_dw.cdc_source_customers
""")

print("✅ CDC pipeline tables created:")
print("   Bronze: cdc_bronze_events (raw CDC events)")
print("   Silver: cdc_silver_customers (current state)")
print(f"   Silver rows: {spark.table('techmart_dw.cdc_silver_customers').count()}")

---
# 📗 Section 3: Simulating CDC Events

## Generating Realistic Change Events

In production, CDC events come from Debezium/Lakeflow Connect reading the database log.
Here we simulate 5 rounds of changes to demonstrate the full CDC lifecycle.

In [0]:
# Round 1: Simulate changes to the source system
print("🔄 ROUND 1: Source System Changes")
print("=" * 60)

# Simulate: Alice moves to San Francisco, Bob gets upgraded
spark.sql("""
INSERT INTO techmart_dw.cdc_bronze_events VALUES
(1, 'U', 1, 'Alice Smith', 'alice@techmart.com', 'San Francisco', 'gold', 5500.00, true, 
    '2024-03-15 10:00:00', 1001, 'lsn_001'),
(2, 'U', 2, 'Bob Jones', 'bob@techmart.com', 'Los Angeles', 'gold', 3000.00, true,
    '2024-03-15 10:01:00', 1002, 'lsn_002'),
(3, 'I', 6, 'Frank Miller', 'frank@techmart.com', 'Denver', 'bronze', 100.00, true,
    '2024-03-15 10:02:00', 1003, 'lsn_003')
""")

print("   Events generated:")
print("   • UPDATE: Alice moved to San Francisco, LTV increased")
print("   • UPDATE: Bob upgraded to gold tier")
print("   • INSERT: New customer Frank")
spark.sql("""
    SELECT event_id, operation, customer_id, name, city, tier, change_timestamp
    FROM techmart_dw.cdc_bronze_events ORDER BY sequence_number
""").show(truncate=False)

In [0]:
# Round 2: More changes including a DELETE
print("🔄 ROUND 2: More Changes (including DELETE)")
print("=" * 60)

spark.sql("""
INSERT INTO techmart_dw.cdc_bronze_events VALUES
(4, 'U', 3, 'Charlie Brown', 'charlie.new@techmart.com', 'Chicago', 'silver', 1200.00, true,
    '2024-03-16 09:00:00', 1004, 'lsn_004'),
(5, 'D', 5, 'Eve Wilson', 'eve@techmart.com', 'Austin', 'silver', 1500.00, false,
    '2024-03-16 09:01:00', 1005, 'lsn_005'),
(6, 'I', 7, 'Grace Hopper', 'grace@techmart.com', 'Boston', 'gold', 7500.00, true,
    '2024-03-16 09:02:00', 1006, 'lsn_006'),
(7, 'U', 1, 'Alice Smith', 'alice@techmart.com', 'San Francisco', 'platinum', 8000.00, true,
    '2024-03-16 09:03:00', 1007, 'lsn_007')
""")

print("   Events generated:")
print("   • UPDATE: Charlie got new email, upgraded to silver")
print("   • DELETE: Eve's account deactivated")
print("   • INSERT: New customer Grace")
print("   • UPDATE: Alice upgraded to platinum (second update!)")
print(f"\n   Total CDC events: {spark.table('techmart_dw.cdc_bronze_events').count()}")

---
# 📗 Section 4: Applying CDC Events with MERGE

## The CDC MERGE Pattern

The key challenge: apply INSERT, UPDATE, and DELETE operations in a single MERGE statement.

```sql
MERGE INTO silver_table AS target
USING (
    -- Deduplicate: keep only the LATEST event per customer
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sequence_number DESC) AS rn
        FROM bronze_events
        WHERE sequence_number > last_processed_sequence
    ) WHERE rn = 1
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.operation = 'D' THEN DELETE
WHEN MATCHED AND source.operation = 'U' THEN UPDATE SET ...
WHEN NOT MATCHED AND source.operation IN ('I', 'U') THEN INSERT ...
```

### Key Considerations

1. **Deduplication**: Multiple events for same key → keep only latest
2. **Ordering**: Use sequence_number (not timestamp) for ordering
3. **Idempotency**: Safe to reprocess the same events
4. **Late arrivals**: Handle events that arrive out of order

In [0]:
# Apply CDC events to Silver table using MERGE
print("🔀 APPLYING CDC EVENTS TO SILVER")
print("=" * 60)

# Show Silver BEFORE
print("\n📌 Silver table BEFORE applying CDC:")
spark.sql("SELECT customer_id, name, city, tier, lifetime_value FROM techmart_dw.cdc_silver_customers ORDER BY customer_id").show()

# Apply CDC: Deduplicate (keep latest per customer), then MERGE
spark.sql("""
MERGE INTO techmart_dw.cdc_silver_customers AS target
USING (
    -- Deduplicate: keep only the LATEST event per customer_id
    SELECT customer_id, name, email, city, tier, lifetime_value, is_active,
           change_timestamp, operation, sequence_number
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY sequence_number DESC
        ) AS rn
        FROM techmart_dw.cdc_bronze_events
    )
    WHERE rn = 1
) AS source
ON target.customer_id = source.customer_id

-- DELETE: Remove records marked for deletion
WHEN MATCHED AND source.operation = 'D' THEN DELETE

-- UPDATE: Apply updates
WHEN MATCHED AND source.operation = 'U' THEN UPDATE SET
    target.name = source.name,
    target.email = source.email,
    target.city = source.city,
    target.tier = source.tier,
    target.lifetime_value = source.lifetime_value,
    target.is_active = source.is_active,
    target.last_modified = source.change_timestamp,
    target._cdc_operation = source.operation,
    target._cdc_timestamp = source.change_timestamp

-- INSERT: Add new records
WHEN NOT MATCHED AND source.operation IN ('I', 'U') THEN INSERT (
    customer_id, name, email, city, tier, lifetime_value, is_active,
    last_modified, _cdc_operation, _cdc_timestamp
) VALUES (
    source.customer_id, source.name, source.email, source.city, source.tier,
    source.lifetime_value, source.is_active, source.change_timestamp,
    source.operation, source.change_timestamp
)
""")

# Show Silver AFTER
print("\n📌 Silver table AFTER applying CDC:")
spark.sql("SELECT customer_id, name, city, tier, lifetime_value, _cdc_operation FROM techmart_dw.cdc_silver_customers ORDER BY customer_id").show()

print("\n✅ CDC Applied Successfully:")
print("   • Alice: NYC → San Francisco → platinum (latest update wins)")
print("   • Bob: silver → gold")
print("   • Charlie: new email, bronze → silver")
print("   • Eve: DELETED")
print("   • Frank: INSERTED (new customer)")
print("   • Grace: INSERTED (new customer)")

---
# 📗 Section 5: Handling CDC Edge Cases

## The Hard Problems in CDC

| Edge Case | Problem | Solution |
|-----------|---------|----------|
| **Out-of-order events** | Event 5 arrives before event 4 | Use sequence_number, not timestamp |
| **Duplicate events** | Same event delivered twice | Deduplicate on (customer_id, sequence_number) |
| **Late arrivals** | Event from yesterday arrives today | Watermark + reprocessing window |
| **Schema changes** | Source adds a column mid-stream | mergeSchema + NULL for missing columns |
| **Deletes then re-inserts** | Customer deleted then re-created | Sequence number determines final state |
| **Bulk operations** | Source does UPDATE WHERE (millions of events) | Batch processing with checkpoints |

## Out-of-Order Events

```
Real order:    Event 1 → Event 2 → Event 3 → Event 4 → Event 5
Arrival order: Event 1 → Event 2 → Event 5 → Event 3 → Event 4
                                    ↑ LATE!    ↑ LATE!

Solution: Always use sequence_number (monotonically increasing) to determine
          the "latest" state, NOT arrival time.
```

## Idempotent CDC Processing

The pipeline must be safe to rerun:
- Track the last processed sequence_number
- On rerun, reprocess from last checkpoint
- MERGE handles duplicates naturally (same key = update, not duplicate)

In [0]:
# Handling out-of-order events
print("⚠️ HANDLING OUT-OF-ORDER CDC EVENTS")
print("=" * 60)

# Simulate out-of-order: event with seq=1010 arrives BEFORE seq=1008
spark.sql("""
INSERT INTO techmart_dw.cdc_bronze_events VALUES
(10, 'U', 6, 'Frank Miller', 'frank@techmart.com', 'Denver', 'silver', 500.00, true,
    '2024-03-17 14:00:00', 1010, 'lsn_010'),
(8, 'U', 6, 'Frank Miller', 'frank@techmart.com', 'Denver', 'bronze', 200.00, true,
    '2024-03-17 10:00:00', 1008, 'lsn_008'),
(9, 'U', 6, 'Frank Miller', 'frank@techmart.com', 'Denver', 'bronze', 350.00, true,
    '2024-03-17 12:00:00', 1009, 'lsn_009')
""")

print("   Inserted 3 events for Frank (out of order):")
print("   • seq=1010 (arrived first): tier=silver, LTV=500")
print("   • seq=1008 (arrived second): tier=bronze, LTV=200")
print("   • seq=1009 (arrived third): tier=bronze, LTV=350")
print()
print("   Correct final state should be seq=1010 (highest sequence)")
print("   NOT seq=1009 (last to arrive)")

# Re-apply CDC with proper deduplication
spark.sql("""
MERGE INTO techmart_dw.cdc_silver_customers AS target
USING (
    SELECT customer_id, name, email, city, tier, lifetime_value, is_active,
           change_timestamp, operation
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY customer_id ORDER BY sequence_number DESC
        ) AS rn
        FROM techmart_dw.cdc_bronze_events
    ) WHERE rn = 1
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.operation = 'D' THEN DELETE
WHEN MATCHED THEN UPDATE SET
    target.name = source.name, target.email = source.email,
    target.city = source.city, target.tier = source.tier,
    target.lifetime_value = source.lifetime_value,
    target.is_active = source.is_active,
    target.last_modified = source.change_timestamp,
    target._cdc_operation = source.operation,
    target._cdc_timestamp = source.change_timestamp
WHEN NOT MATCHED AND source.operation != 'D' THEN INSERT (
    customer_id, name, email, city, tier, lifetime_value, is_active,
    last_modified, _cdc_operation, _cdc_timestamp
) VALUES (
    source.customer_id, source.name, source.email, source.city, source.tier,
    source.lifetime_value, source.is_active, source.change_timestamp,
    source.operation, source.change_timestamp
)
""")

# Verify Frank has the correct state (seq=1010, tier=silver)
print("\n✅ Frank's state after handling out-of-order events:")
spark.sql("SELECT customer_id, name, tier, lifetime_value FROM techmart_dw.cdc_silver_customers WHERE customer_id = 6").show()
print("   Correct! tier=silver, LTV=500 (from seq=1010, the highest sequence)")


---
# 📗 Section 6: Delta Change Data Feed (CDF)

## Reading Changes FROM a Delta Table

While CDC captures changes from SOURCE systems, Delta's Change Data Feed (CDF)
lets you read changes FROM your Delta tables — useful for downstream consumers.

```
Source DB → CDC → Bronze → Silver (CDF enabled) → Gold reads CDF → Aggregates
                                    ↑
                              table_changes() lets Gold
                              read only what changed in Silver
```

### CDF vs CDC

| Aspect | CDC (from source) | CDF (from Delta table) |
|--------|-------------------|----------------------|
| **Direction** | Source → Delta | Delta → Downstream |
| **Tool** | Debezium, Lakeflow Connect | Built into Delta Lake |
| **Enable** | Configure connector | `delta.enableChangeDataFeed = true` |
| **Query** | Read from Kafka/Bronze | `table_changes('table', start, end)` |
| **Use case** | Ingest changes | Incremental downstream processing |

In [0]:
# Query CDF from our Silver table
print("📡 DELTA CHANGE DATA FEED (CDF)")
print("=" * 60)

# CDF was enabled when we created the Silver table
# Let's see what changes were recorded

# Get the version range
history = spark.sql("DESCRIBE HISTORY techmart_dw.cdc_silver_customers").collect()
max_version = history[0]["version"]
print(f"   Table versions: 0 to {max_version}")

# Query changes
print(f"\n📌 Changes from version 1 to {max_version}:")
try:
    changes = spark.sql(f"""
        SELECT customer_id, name, tier, _change_type, _commit_version
        FROM table_changes('techmart_dw.cdc_silver_customers', 1, {max_version})
        ORDER BY _commit_version, customer_id
    """)
    changes.show(truncate=False)
    
    # Summarize changes
    print("\n📊 Change Summary:")
    spark.sql(f"""
        SELECT _change_type, COUNT(*) as count
        FROM table_changes('techmart_dw.cdc_silver_customers', 1, {max_version})
        GROUP BY _change_type
    """).show()
except Exception as e:
    print(f"   ⚠️ CDF query: {e}")
    print("   (CDF captures changes made AFTER enablement)")

---
# 📗 Section 7: Practice Exercises

## Exercise 1: Build a CDC Processor

In [0]:
# ============================================
# 🎯 YOUR TURN: Build an Incremental CDC Processor
# ============================================
# Build a function that:
# 1. Tracks the last processed sequence_number
# 2. Only processes NEW events (since last run)
# 3. Deduplicates events per customer_id
# 4. Applies changes via MERGE
# 5. Updates the checkpoint
#
# This makes the pipeline INCREMENTAL and IDEMPOTENT.
#
# Write your processor below:


In [0]:
# ============================================
# ✅ SOLUTION: Incremental CDC Processor
# ============================================

class IncrementalCDCProcessor:
    """
    Production-style incremental CDC processor.
    Tracks checkpoints for incremental processing.
    Idempotent — safe to rerun.
    """
    
    def __init__(self, bronze_table, silver_table, key_column="customer_id"):
        self.bronze_table = bronze_table
        self.silver_table = silver_table
        self.key_column = key_column
        self.checkpoint_table = f"{silver_table}_checkpoint"
        self._ensure_checkpoint_table()
    
    def _ensure_checkpoint_table(self):
        """Create checkpoint table if it doesn't exist."""
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.checkpoint_table} (
                last_sequence_number BIGINT,
                last_processed_at TIMESTAMP,
                records_processed INT
            ) USING DELTA
        """)
        # Initialize if empty
        if spark.table(self.checkpoint_table).count() == 0:
            spark.sql(f"INSERT INTO {self.checkpoint_table} VALUES (0, current_timestamp(), 0)")
    
    def get_last_checkpoint(self):
        """Get the last processed sequence number."""
        row = spark.sql(f"SELECT MAX(last_sequence_number) AS seq FROM {self.checkpoint_table}").collect()[0]
        return row["seq"] or 0
    
    def process(self):
        """Process new CDC events since last checkpoint."""
        last_seq = self.get_last_checkpoint()
        
        # Count new events
        new_events_count = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {self.bronze_table}
            WHERE sequence_number > {last_seq}
        """).collect()[0]["cnt"]
        
        if new_events_count == 0:
            print(f"   ⏭️ No new events (last processed: seq={last_seq})")
            return 0
        
        print(f"   📥 New events to process: {new_events_count} (since seq={last_seq})")
        
        # Apply CDC with deduplication
        spark.sql(f"""
            MERGE INTO {self.silver_table} AS target
            USING (
                SELECT {self.key_column}, name, email, city, tier, lifetime_value, 
                       is_active, change_timestamp, operation, sequence_number
                FROM (
                    SELECT *, ROW_NUMBER() OVER (
                        PARTITION BY {self.key_column} ORDER BY sequence_number DESC
                    ) AS rn
                    FROM {self.bronze_table}
                    WHERE sequence_number > {last_seq}
                ) WHERE rn = 1
            ) AS source
            ON target.{self.key_column} = source.{self.key_column}
            WHEN MATCHED AND source.operation = 'D' THEN DELETE
            WHEN MATCHED THEN UPDATE SET
                target.name = source.name, target.email = source.email,
                target.city = source.city, target.tier = source.tier,
                target.lifetime_value = source.lifetime_value,
                target.is_active = source.is_active,
                target.last_modified = source.change_timestamp,
                target._cdc_operation = source.operation,
                target._cdc_timestamp = source.change_timestamp
            WHEN NOT MATCHED AND source.operation != 'D' THEN INSERT (
                {self.key_column}, name, email, city, tier, lifetime_value, is_active,
                last_modified, _cdc_operation, _cdc_timestamp
            ) VALUES (
                source.{self.key_column}, source.name, source.email, source.city,
                source.tier, source.lifetime_value, source.is_active,
                source.change_timestamp, source.operation, source.change_timestamp
            )
        """)
        
        # Update checkpoint
        max_seq = spark.sql(f"""
            SELECT MAX(sequence_number) AS seq FROM {self.bronze_table}
            WHERE sequence_number > {last_seq}
        """).collect()[0]["seq"]
        
        spark.sql(f"""
            INSERT INTO {self.checkpoint_table} 
            VALUES ({max_seq}, current_timestamp(), {new_events_count})
        """)
        
        print(f"   ✅ Processed {new_events_count} events")
        print(f"   📍 Checkpoint updated: seq={max_seq}")
        return new_events_count


# Demo the incremental processor
print("🔄 INCREMENTAL CDC PROCESSOR DEMO")
print("=" * 60)

processor = IncrementalCDCProcessor(
    bronze_table="techmart_dw.cdc_bronze_events",
    silver_table="techmart_dw.cdc_silver_customers"
)

# First run (processes all events)
print("\n--- RUN 1 (first run) ---")
processor.process()

# Second run (no new events)
print("\n--- RUN 2 (no new events) ---")
processor.process()

# Add new events and run again
print("\n--- Adding new events ---")
spark.sql("""
    INSERT INTO techmart_dw.cdc_bronze_events VALUES
    (11, 'U', 2, 'Bob Jones', 'bob.vip@techmart.com', 'Los Angeles', 'platinum', 10000.00, true,
        '2024-03-18 10:00:00', 1011, 'lsn_011')
""")

print("\n--- RUN 3 (processes only new event) ---")
processor.process()

# Show final state
print("\n📌 Final Silver table state:")
spark.sql("SELECT customer_id, name, tier, lifetime_value FROM techmart_dw.cdc_silver_customers ORDER BY customer_id").show()


---
# 📗 Section 8: Summary & Quick Reference

## CDC Pipeline Cheat Sheet

```
CDC CAPTURE METHODS:
  Log-based (best): Read DB transaction log → Debezium, Lakeflow Connect
  Trigger-based: DB triggers write to change table
  Timestamp-based: WHERE modified_at > last_run (misses deletes!)

CDC EVENT STRUCTURE:
  operation: I (insert), U (update), D (delete)
  before: old values (NULL for inserts)
  after: new values (NULL for deletes)
  sequence_number: ordering guarantee (use this, not timestamp!)

BRONZE LAYER (raw CDC events):
  Append-only (never modify!)
  Keep ALL events (audit trail)
  Include: operation, sequence_number, source metadata

SILVER LAYER (current state):
  Apply CDC via MERGE
  Deduplicate: ROW_NUMBER() OVER (PARTITION BY key ORDER BY seq DESC)
  Handle all operations: INSERT, UPDATE, DELETE

MERGE PATTERN FOR CDC:
  MERGE INTO silver USING (deduplicated_source)
  ON silver.key = source.key
  WHEN MATCHED AND op = 'D' THEN DELETE
  WHEN MATCHED AND op = 'U' THEN UPDATE SET *
  WHEN NOT MATCHED AND op IN ('I','U') THEN INSERT *

EDGE CASES:
  Out-of-order: Use sequence_number (not timestamp) for ordering
  Duplicates: Deduplicate on (key, sequence_number)
  Late arrivals: Reprocessing window + idempotent MERGE
  Schema changes: mergeSchema=true + NULL for missing columns

DELTA CHANGE DATA FEED (CDF):
  Enable: ALTER TABLE SET TBLPROPERTIES ('delta.enableChangeDataFeed'='true')
  Query: SELECT * FROM table_changes('table', start_version, end_version)
  Columns: _change_type, _commit_version, _commit_timestamp

INCREMENTAL PROCESSING:
  Track last_processed_sequence_number in checkpoint table
  Only process events WHERE seq > last_checkpoint
  Update checkpoint after successful MERGE
  Idempotent: safe to rerun (MERGE handles duplicates)
```

## CDC vs SCD — When to Use Each

| Aspect | CDC (this notebook) | SCD (notebook 05) |
|--------|--------------------|--------------------|
| **Focus** | Capturing changes from source | Storing dimension history |
| **Input** | Source database transaction log | CDC events or full loads |
| **Output** | Current state (Silver table) | Versioned history (Type 2) |
| **Pattern** | MERGE (upsert + delete) | MERGE with effective dates |
| **Use together** | CDC captures changes → SCD stores history |

## Next Steps

- **Notebook 05:** SCD Masterclass (apply CDC to build dimension history)
- **Notebook 46:** Delta Lake Internals (OPTIMIZE, VACUUM for CDC tables)
- **Notebook 24:** Streaming Processing (real-time CDC with Structured Streaming)
- **Notebook 04:** Medallion Architecture (Bronze/Silver/Gold patterns)

---
*Notebook 48 of the Databricks Data Engineering series*
*Study Order: 45 (Advanced Topics)*

---
# 📗 Section 9: CDC in Production

## Production CDC Architecture

```
Source DB (PostgreSQL)
    |
    | (Debezium reads transaction log)
    v
Kafka Topic: orders-cdc
    |
    | (Spark Structured Streaming)
    v
Bronze: raw_cdc_events (append-only, all events)
    |
    | (MERGE with deduplication)
    v
Silver: current_orders (current state)
    |
    | (Aggregation)
    v
Gold: daily_metrics
```

## CDC vs Full Load -- When to Use Each

| Scenario | Use CDC | Use Full Load |
|----------|---------|---------------|
| Large table (>1M rows) | Yes | No (too slow) |
| Need deletes captured | Yes | No (can't detect) |
| Low latency required | Yes | No |
| Small table (<10K rows) | No | Yes (simpler) |
| No reliable change tracking | No | Yes |
| Initial historical load | No | Yes (then switch to CDC) |

## Handling CDC Edge Cases

```python
# Edge case 1: Multiple updates in same batch
# Solution: Deduplicate, keep latest by sequence_number
deduped = (cdc_events
    .withColumn("rn", row_number().over(
        Window.partitionBy("id").orderBy(col("sequence_number").desc())
    ))
    .filter(col("rn") == 1)
    .drop("rn"))

# Edge case 2: Delete then re-insert
# Solution: sequence_number determines final state
# If delete (seq=100) comes after insert (seq=99), result is deleted
# If insert (seq=101) comes after delete (seq=100), result is inserted

# Edge case 3: Schema change in source
# Solution: Use mergeSchema=True in Bronze, handle in Silver
```

In [0]:
# CDC production patterns summary
print("CDC Production Patterns")
print("=" * 60)

cdc_patterns = {
    "Capture Methods": {
        "Log-based (best)": "Read DB transaction log -- Debezium, Lakeflow Connect",
        "Trigger-based": "DB triggers write to change table -- simple but overhead",
        "Timestamp-based": "WHERE modified_at > last_run -- misses deletes!",
    },
    "Event Structure": {
        "operation": "I (insert), U (update), D (delete)",
        "before": "Old values (NULL for inserts)",
        "after": "New values (NULL for deletes)",
        "sequence_number": "Ordering guarantee (use this, not timestamp)",
    },
    "Bronze Layer": {
        "pattern": "Append-only (never modify!)",
        "purpose": "Audit trail, reprocessing capability",
        "key_columns": "operation, sequence_number, _ingested_at",
    },
    "Silver Layer": {
        "pattern": "MERGE with deduplication",
        "dedup": "ROW_NUMBER() OVER (PARTITION BY id ORDER BY seq DESC)",
        "merge": "WHEN MATCHED AND op=D THEN DELETE, WHEN MATCHED THEN UPDATE, WHEN NOT MATCHED THEN INSERT",
    },
    "Incremental Processing": {
        "checkpoint": "Track last_processed_sequence_number",
        "idempotent": "MERGE handles duplicates naturally",
        "reprocess": "Reset checkpoint to reprocess from any point",
    },
}

for category, details in cdc_patterns.items():
    print(f"\n{category}:")
    for key, value in details.items():
        print(f"  {key}: {value}")


In [0]:
# Cleanup
print("🧹 CLEANUP")
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_source_customers")
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_bronze_events")
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_silver_customers")
spark.sql("DROP TABLE IF EXISTS techmart_dw.cdc_silver_customers_checkpoint")
print("✅ CDC demo tables cleaned up")